# Community Crime Classifier - Exploratory Data Analysis

Exploring crime patterns across Calgary communities to inform classification models.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import sys
sys.path.insert(0, '..')
from src.data_loader import (
    load_or_fetch_crime_data, load_or_fetch_census_data,
    preprocess_crime_data, preprocess_census_data,
    create_community_features, create_temporal_crime_data
)

In [ ]:
crime_df = load_or_fetch_crime_data('../data', limit=100000)
crime_df = preprocess_crime_data(crime_df)
print(f'Crime data shape: {crime_df.shape}')
crime_df.head()

In [ ]:
print('Crime categories:', crime_df['category'].nunique())
print(crime_df['category'].value_counts())

In [ ]:
# Total crimes by category
cat_totals = crime_df.groupby('category')['crime_count'].sum().sort_values()
fig = px.bar(x=cat_totals.values, y=cat_totals.index, orientation='h',
             title='Total Crime by Category', color=cat_totals.values,
             color_continuous_scale='Reds')
fig.update_layout(height=500, showlegend=False)
fig.show()

In [ ]:
# Yearly trend
yearly = crime_df.groupby('year')['crime_count'].sum().reset_index()
fig = px.line(yearly, x='year', y='crime_count', title='Annual Crime Trend', markers=True)
fig.show()

In [ ]:
# Monthly seasonality
monthly = crime_df.groupby('month')['crime_count'].sum().reset_index()
fig = px.bar(monthly, x='month', y='crime_count', title='Crime by Month',
             color='crime_count', color_continuous_scale='YlOrRd')
fig.show()

In [ ]:
# Community analysis
try:
    census_df = load_or_fetch_census_data('../data', limit=100000)
    census_df = preprocess_census_data(census_df)
except Exception:
    census_df = pd.DataFrame()

community_df = create_community_features(crime_df, census_df)
print(f'Communities: {len(community_df)}')
community_df.describe()

In [ ]:
# Top communities by crime
top20 = community_df.nlargest(20, 'total_crimes')
fig = px.bar(top20, x='community', y='total_crimes',
             title='Top 20 Communities by Crime Count')
fig.update_layout(xaxis_tickangle=-45)
fig.show()